# TML Assignment 3 - Robustness
## resnet18 + TRADES + AWP + EMA + SGDR-4 [40,60,80,100] (100 epochs)

**Method change (the genuine frontier-expander): Adversarial Weight Perturbation**
(Wu et al. NeurIPS 2020) on top of the AA-honest TRADES base. Before each SGD update,
AWP perturbs the WEIGHTS to the loss-maximising point in a gamma-ball, takes the
gradient there, then restores - flattening the weight-loss landscape. Unlike FAT/MART
(which inflate the CE-PGD proxy), AWP raises true (AutoAttack) robustness and is the
standard no-extra-data robust booster on RobustBench.

**Why this run.** AutoAttack showed MART's robust edge was mostly CE-PGD masking
(0.50 -> 0.445 AA), no better than TRADES (~0.457). The true robust axis is the real
cap; AWP is the one method with evidence of raising it genuinely. Our earlier AWP
failed from a buggy update that moved weights by gamma*||w|| every step; THIS uses the
official proxy form where perturb/restore cancel exactly, so only the SGD step moves
the weights. Warmup 10 epochs, gamma=0.005, beta=3 (proven). 100ep to plateau.

**Verify with AutoAttack before submitting** (code/eval_autoattack.py) - TRADES+AWP
is expected to be the more genuine (server-faithful) of the two runs.

Invariants kept: EMA0.999, fp32 KL +-30 clamp, crop+flip, best-ckpt by val_score,
48k/2k split, lr0.1 warmup3, eps8/255. ~1.7x slower/epoch than plain TRADES (AWP adds
a proxy forward+backward). Run cells top to bottom.

## 1. Setup

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/tml_assignment3'
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoint dir:', CKPT_DIR)

## 2. Download dataset

In [ ]:
DATA_PATH = '/content/train.npz'
if not os.path.exists(DATA_PATH):
    !wget -q -O {DATA_PATH} "https://huggingface.co/datasets/SprintML/tml26_task3/resolve/main/train.npz"
print('Downloaded:', os.path.exists(DATA_PATH), os.path.getsize(DATA_PATH) / 1e6, 'MB')

## 3. Imports & hyperparameters

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.models import resnet18
import torchvision.transforms.v2 as T
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True  # speeds up conv kernels for fixed input size

# ---- Hyperparameters (proven MART recipe, ported to resnet18 / 80 epochs) ----
NUM_CLASSES = 9
MODEL_NAME = 'resnet18'
BATCH_SIZE = 128
EPOCHS = 100
WARMUP_EPOCHS = 3
VAL_SIZE = 2000
SEED = 42

# Optimizer + LR schedule: warmup -> cosine, peak lr=0.1 (confirmed better than 0.02).
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

# PGD attack (training) - CE-based, identical to all previous experiments (Madry 2018).
EPS = 8 / 255
ALPHA = 2 / 255
PGD_STEPS_TRAIN = 7

# PGD attack (evaluation - stronger)
PGD_STEPS_EVAL = 20

# TRADES (Zhang et al. 2019): loss = CE(clean) + TRADES_BETA * KL(clean||adv).
# beta=3.0 is the project-proven optimum (the 0.594 LB recipe).
TRADES_BETA = 3.0

# AWP (Adversarial Weight Perturbation, Wu et al. 2020): before each update, perturb
# the weights to the loss-MAXIMISING point within a gamma-ball, take the gradient
# there, then RESTORE - flattens the weight-loss landscape and genuinely improves
# robust generalisation (raises true/AutoAttack robustness, not just CE-PGD). The
# perturbation cancels exactly (perturb +g*d, step, restore -g*d) so it only changes
# the GRADIENT, never swamps the SGD step (the bug in our earlier AWP attempt).
AWP_GAMMA = 0.005
AWP_WARMUP = 10      # no AWP for the first AWP_WARMUP epochs (BN/weights settle first)

# EMA of all floating-point params/buffers (incl. BatchNorm running stats), every step.
# Validation/checkpointing/submission all use ema_model.
EMA_DECAY = 0.999

# SGDR-3 (the EXACT schedule behind the 0.594 TRADES r18 model): 40-epoch initial
# cosine (lets clean fully converge), then two 20-epoch warm restarts at 60 and 80.
LR_CYCLE_BOUNDARIES = [40, 60, 80, 100]

# Mixed precision (PGD steps dominate per-batch cost)
USE_AMP = (device.type == 'cuda')

RESUME = True
CKPT_PATH = os.path.join(CKPT_DIR, f'{MODEL_NAME}_r18_trades_awp.pt')
BEST_CKPT_PATH = os.path.join(CKPT_DIR, f'{MODEL_NAME}_r18_trades_awp_best.pt')

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Device:', device, '| AMP:', USE_AMP)

## 4. Data loading

In [ ]:
data = np.load(DATA_PATH)
images = torch.from_numpy(data['images']).float() / 255.0  # (N, 3, 32, 32) in [0,1]
labels = torch.from_numpy(data['labels']).long()
print('Images:', images.shape, 'Labels:', labels.shape)

full_dataset = TensorDataset(images, labels)
train_size = len(full_dataset) - VAL_SIZE
train_set, val_set = random_split(
    full_dataset, [train_size, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED)
)
print('Train size:', len(train_set), 'Val size:', len(val_set))

# Standard CIFAR-style augmentation (applied to the clean image before PGD attack)
augment = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)

## 5. Model

In [ ]:
def build_model():
    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

model = build_model().to(device)

ema_model = build_model().to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)
ema_model.eval()

# sanity check matching task_template.py
model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 32, 32, device=device))
assert out.shape == (1, NUM_CLASSES), out.shape
print('Output shape OK:', out.shape)

## 6. PGD attack (identical to all previous PGD-AT experiments)

In [ ]:
def pgd_attack(model, x, y, eps, alpha, steps, random_start=True):
    """Untargeted L_inf PGD attack maximizing cross-entropy. x is in [0,1]. Model is
    set to eval() during attack generation so BatchNorm uses running statistics."""
    was_training = model.training
    model.eval()

    x_orig = x.detach()
    if random_start:
        delta = torch.empty_like(x_orig).uniform_(-eps, eps)
        x_adv = torch.clamp(x_orig + delta, 0.0, 1.0).detach()
    else:
        x_adv = x_orig.clone()

    for _ in range(steps):
        x_adv.requires_grad_(True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x_adv)
            loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        with torch.no_grad():
            x_adv = x_adv + alpha * grad.sign()
            x_adv = torch.clamp(x_adv, x_orig - eps, x_orig + eps)
            x_adv = torch.clamp(x_adv, 0.0, 1.0)

    if was_training:
        model.train()
    return x_adv.detach()


def fgsm_attack(model, x, y, eps):
    was_training = model.training
    model.eval()
    x_orig = x.detach().requires_grad_(True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        logits = model(x_orig)
        loss = F.cross_entropy(logits, y)
    grad = torch.autograd.grad(loss, x_orig)[0]
    x_adv = torch.clamp(x_orig + eps * grad.sign(), 0.0, 1.0).detach()
    if was_training:
        model.train()
    return x_adv

# ---- TRADES loss (fp32, logits clamped +-30 for fp16 nan-safety) ----
def trades_loss_fn(logits_clean, logits_adv, y, beta):
    lc = logits_clean.float().clamp(-30, 30)
    la = logits_adv.float().clamp(-30, 30)
    loss_natural = F.cross_entropy(lc, y)
    loss_robust = F.kl_div(F.log_softmax(la, dim=1), F.softmax(lc, dim=1), reduction="batchmean")
    return loss_natural + beta * loss_robust


# ---- AWP (Wu et al. 2020), faithful proxy implementation (csdongxian/AWP) ----
AWP_EPS = 1e-8

def diff_in_weights(model, proxy):
    diff = {}
    msd, psd = model.state_dict(), proxy.state_dict()
    for (k, w_m), (_, w_p) in zip(msd.items(), psd.items()):
        if len(w_m.size()) <= 1 or "weight" not in k:
            continue
        dw = w_p - w_m
        diff[k] = w_m.norm() / (dw.norm() + AWP_EPS) * dw   # per-layer, scaled to weight norm
    return diff

def add_into_weights(model, diff, coeff=1.0):
    with torch.no_grad():
        for name, p in model.named_parameters():
            if name in diff:
                p.add_(coeff * diff[name])

class AWP:
    def __init__(self, model, proxy, proxy_optim, gamma):
        self.model = model; self.proxy = proxy
        self.proxy_optim = proxy_optim; self.gamma = gamma
    def calc_awp(self, x_adv, x_clean, y, beta):
        self.proxy.load_state_dict(self.model.state_dict())
        self.proxy.train()
        lc = self.proxy(x_clean); la = self.proxy(x_adv)
        loss = -trades_loss_fn(lc, la, y, beta)   # gradient ASCENT on the TRADES loss
        self.proxy_optim.zero_grad(); loss.backward(); self.proxy_optim.step()
        return diff_in_weights(self.model, self.proxy)
    def perturb(self, diff):
        add_into_weights(self.model, diff, coeff=1.0 * self.gamma)
    def restore(self, diff):
        add_into_weights(self.model, diff, coeff=-1.0 * self.gamma)


## 7. Optimizer, LR schedule (warmup + SGDR cosine), training loop

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

# AWP proxy network + its own optimizer (lr=0.01 is the standard AWP ascent step)
proxy = build_model().to(device)
proxy_optim = torch.optim.SGD(proxy.parameters(), lr=0.01)
awp = AWP(model, proxy, proxy_optim, gamma=AWP_GAMMA)


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    # SGDR-style cosine restarts. With LR_CYCLE_BOUNDARIES=[40,60,80]: a 40-epoch
    # cosine, then jump back up for a 20-epoch cosine (40-59), then another
    # 20-epoch cosine (60-79). The trailing default avoids StopIteration on the
    # final scheduler.step() at epoch == last boundary.
    total = next((b for b in LR_CYCLE_BOUNDARIES if epoch < b), LR_CYCLE_BOUNDARIES[-1])
    progress = (epoch - WARMUP_EPOCHS) / max(1, total - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

start_epoch = 0
best_score = -1.0
if RESUME and os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    ema_model.load_state_dict(ckpt['ema_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from epoch {start_epoch}')
else:
    print('Starting fresh training')

if RESUME and os.path.exists(BEST_CKPT_PATH):
    best_score = torch.load(BEST_CKPT_PATH, map_location='cpu').get('score', -1.0)
    print(f'Loaded best score so far: {best_score:.4f}')

In [ ]:
@torch.no_grad()
def evaluate_clean(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total


def evaluate_pgd(model, loader, eps, alpha, steps, max_batches=None):
    model.eval()
    correct, total = 0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        x_adv = pgd_attack(model, x, y, eps, alpha, steps)
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x_adv).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

In [ ]:
@torch.no_grad()
def update_ema(ema_model, model, decay):
    msd = model.state_dict()
    for k, v in ema_model.state_dict().items():
        if v.dtype.is_floating_point:
            v.mul_(decay).add_(msd[k], alpha=1 - decay)
        else:
            v.copy_(msd[k])


for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0 = time.time()
    running_loss, running_correct, running_total = 0.0, 0, 0
    use_awp = epoch >= AWP_WARMUP

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        x = augment(x)

        # CE-based PGD-7 adversarial examples (proven inner attack)
        x_adv = pgd_attack(model, x, y, EPS, ALPHA, PGD_STEPS_TRAIN)

        model.train()
        # AWP: perturb weights to the loss-maximising point (after warmup)
        if use_awp:
            diff = awp.calc_awp(x_adv, x, y, TRADES_BETA)
            awp.perturb(diff)

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits_clean = model(x)
            logits_adv = model(x_adv)
        loss = trades_loss_fn(logits_clean, logits_adv, y, TRADES_BETA)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if use_awp:
            awp.restore(diff)   # exact cancellation -> weight only moved by the SGD step

        update_ema(ema_model, model, EMA_DECAY)

        running_loss += loss.item() * y.size(0)
        running_correct += (logits_adv.argmax(dim=1) == y).sum().item()
        running_total += y.size(0)
        pbar.set_postfix(loss=running_loss / running_total, adv_acc=running_correct / running_total)

    scheduler.step()

    train_loss = running_loss / running_total
    train_adv_acc = running_correct / running_total
    val_clean_acc = evaluate_clean(ema_model, val_loader)
    val_pgd_acc = evaluate_pgd(ema_model, val_loader, EPS, ALPHA, PGD_STEPS_TRAIN, max_batches=5)
    val_score = 0.5 * val_clean_acc + 0.5 * val_pgd_acc

    dt = time.time() - t0
    print(f"Epoch {epoch+1}/{EPOCHS} | lr {scheduler.get_last_lr()[0]:.4f} | "
          f"train_loss {train_loss:.4f} | train_adv_acc {train_adv_acc:.4f} | "
          f"val_clean_acc(ema) {val_clean_acc:.4f} | val_pgd7_acc(ema) {val_pgd_acc:.4f} | "
          f"val_score(ema) {val_score:.4f} | awp {use_awp} | {dt:.1f}s")

    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "ema_state_dict": ema_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "val_clean_acc": val_clean_acc,
        "val_pgd7_acc": val_pgd_acc,
        "score": val_score,
    }
    torch.save(ckpt, CKPT_PATH)

    if val_score > best_score:
        best_score = val_score
        torch.save(ckpt, BEST_CKPT_PATH)
        print(f"  -> new best (val_score={val_score:.4f}), saved to {BEST_CKPT_PATH}")


## 8. Final evaluation (best checkpoint, stronger PGD-20)

In [ ]:
# Load the best checkpoint by val_score (not necessarily the last epoch)
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
model.load_state_dict(best_ckpt['ema_state_dict'])
print(f"Loaded best checkpoint (EMA weights) from epoch {best_ckpt['epoch']+1} (val_score={best_ckpt['score']:.4f})")

final_clean_acc = evaluate_clean(model, val_loader)
final_pgd20_acc = evaluate_pgd(model, val_loader, EPS, ALPHA, PGD_STEPS_EVAL)

model.eval()
fgsm_correct, fgsm_total = 0, 0
for x, y in val_loader:
    x, y = x.to(device), y.to(device)
    x_adv = fgsm_attack(model, x, y, EPS)
    with torch.no_grad():
        pred = model(x_adv).argmax(dim=1)
    fgsm_correct += (pred == y).sum().item()
    fgsm_total += y.size(0)
final_fgsm_acc = fgsm_correct / fgsm_total

print(f'Final clean accuracy:  {final_clean_acc:.4f}')
print(f'Final FGSM accuracy:   {final_fgsm_acc:.4f} (eps={EPS:.4f})')
print(f'Final PGD-20 accuracy: {final_pgd20_acc:.4f} (eps={EPS:.4f}, alpha={ALPHA:.4f})')
print(f'Estimated score (0.5*clean + 0.5*pgd20): {0.5*final_clean_acc + 0.5*final_pgd20_acc:.4f}')

## 9. Save submission state dict

In [ ]:
SUBMIT_PATH = os.path.join(PROJECT_DIR, f'{MODEL_NAME}_r18_trades_awp_submission.pt')

# sanity checks matching task_template.py / submission.py requirements
model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 32, 32, device=device))
assert out.shape == (1, NUM_CLASSES)

torch.save(model.state_dict(), SUBMIT_PATH)
print('Saved submission state dict to:', SUBMIT_PATH)
print('Model name for submission.py:', MODEL_NAME)